# Final Project – Task 1: Data Exploration & Service Idea
**Data Literacy Foundations · 2026 Fall**

**Service proposal:** Cosmetics ingredient finder and product comparer. A user selects their skin type, ingredients to include/avoid, and a price ceiling; the prototype filters real product records and compares ingredient sets.

**Data integrity:** All statistics, samples, tables, graphs, observations and prototype examples in the PDF are calculated **only after downloading the actual CSV**. Do not submit a report that has not been generated by running this notebook.

**Before submission:** Edit `CONFIG` (confirm Group 5 member names/IDs, paste the GitHub URL, verify license/provenance). Use `Kernel → Restart & Run All` to create `outputs/Task1_Report.pdf` (3 pages), chart, summary CSV and `results.json`. If downloads fail, manually download Kaggle CSV and set `COSMETICS_CSV` (and optionally `COSMETICS_B_CSV`) to the local path.

**Important limitation:** A binary skin-type flag from a cosmetics catalog is metadata, **not a clinical safety or suitability judgment**. Likewise, the absence of a term in a parsed list is not proof that a product is free of that substance. Ingredient names require further normalization.


In [ ]:
import os, re, ast, json, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------------------------------------------------
# EDIT THESE BEFORE SUBMITTING (they are printed in the PDF report)
# Verify every TODO value on the Kaggle dataset pages.
# ----------------------------------------------------------------------
CONFIG = {
    "team_name": "Group 5",                                   # TODO
    # [name, student ID, optional romanized name]. If the computer has no Korean font, the PDF shows the
    # romanized name (3rd item) instead of the Hangul name, so fill it in if you see a warning.
    "team": [["Huishin Moon", "20260279"],
             ["성현아", "20260566", None],
             ["문지원", "20260435", None],
             ["조윤서", "20260480", None]],
    "github_url": "[To be added after creating the team repository]",          # TODO
    "dataset_a": {                                             # SELECTED dataset
        "name": "Cosmetics Datasets (Kaggle: kingabzpro)",
        "kaggle_slug": "kingabzpro/cosmetics-datasets",        # TODO: confirm the slug on Kaggle
        "url": "https://www.kaggle.com/datasets/kingabzpro/cosmetics-datasets",
        "license": "Not independently verified; check Kaggle dataset Terms/License",       # TODO
        "provenance": "Snapshot of Sephora product listings; collection date not confirmed",  # TODO
    },
    "dataset_b": {                                             # second candidate
        "name": "Skincare Products and their Ingredients (Kaggle: Erin Ward)",
        "kaggle_slug": "eward96/skincare-products-clean-dataset",
        "url": "https://www.kaggle.com/datasets/eward96/skincare-products-clean-dataset",
        "record": "One skincare product (name, retailer URL, product type, cleaned ingredient list, listed GBP price)",
        "size": "Over 1,000 products and 5 columns (Kaggle description); count only if downloaded",
        "service": "Ingredient-based product search by product type and price",
        "access": "Kaggle download; verify license and original reuse rights before redistribution",
        "limitations": "Single-retailer catalog; no skin-type flags/ratings; listed prices may change and "
                       "licensing for redistribution needs confirmation.",
    },
}
# Metadata was checked against the accessible Kaggle pages and a secondary license index.
# Confirm candidate A's license and provenance with the owner before redistributing data.
R = {"config": CONFIG}   # R collects every number that goes into the PDF report

## Step 2 – Load and inspect the selected dataset

In [ ]:
def read_csv_any(path):
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, low_memory=False, encoding="latin-1")

def load_kaggle_csv(slug, env_name=None):
    """Download with kagglehub (or read the CSV named in an environment variable)."""
    local = os.environ.get(env_name) if env_name else None
    if local:
        return read_csv_any(local), local
    import kagglehub
    folder = kagglehub.dataset_download(slug)
    csvs = sorted(os.path.join(r, f) for r, _, fs in os.walk(folder) for f in fs if f.lower().endswith(".csv"))
    assert csvs, f"No CSV found in {folder}"
    best = None
    for p in csvs:                       # prefer the file that has an ingredients column
        if any("ingred" in c.lower() for c in pd.read_csv(p, nrows=0).columns):
            best = p
            break
    best = best or csvs[0]
    return read_csv_any(best), best

df, csv_path = load_kaggle_csv(CONFIG["dataset_a"]["kaggle_slug"], "COSMETICS_CSV")
# The selected Kaggle catalog contains numeric prices; coerce carefully if the file is edited.
for c in df.columns:
    if c.lower() in ("price", "rank", "rating"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
print("Loaded:", csv_path)

# Map the dataset's column names to the roles we need (case-insensitive)
low = {c.lower(): c for c in df.columns}
def pick(*names):
    return next((low[n] for n in names if n in low), None)

COL = dict(name=pick("name", "product_name"), brand=pick("brand"),
           category=pick("label", "category", "product_type", "type"),
           price=pick("price"), rating=pick("rank", "rating"),
           ingredients=pick("ingredients", "clean_ingreds", "ingredient"))
SKIN = [low[c] for c in ["combination", "dry", "normal", "oily", "sensitive"] if c in low]
print("Detected columns:", COL, "| skin-type columns:", SKIN)
assert all(COL[k] for k in ("name", "category", "ingredients", "price")), f"Need product name/category/ingredients/price; found {list(df.columns)}"
R["missing_required"] = [k for k in ["name", "category", "price", "ingredients"] if not COL[k]]

n_rows, n_cols = df.shape
print(f"{n_rows:,} rows x {n_cols} columns")
print(df.dtypes)

In [ ]:
# Representative records
df.head()

In [ ]:
R["meta"] = {
    "n_rows": int(n_rows), "n_cols": int(n_cols), "columns": [str(c) for c in df.columns],
    "dtype_counts": {str(k): int(v) for k, v in df.dtypes.value_counts().items()},
    "n_categories": int(df[COL["category"]].nunique()) if COL["category"] else 0,
    "n_brands": int(df[COL["brand"]].nunique()) if COL["brand"] else 0,
}

MEANING = {
    "name": "Product name", "brand": "Brand", "category": "Product category",
    "price": "Listed price (currency not confirmed)",
    "rating": "Rating / rank value (scale not documented)",
    "ingredients": "Ingredient list stored as one text value",
}
key_rows = []
for role in ["name", "brand", "category", "price", "rating", "ingredients"]:
    c = COL[role]
    if c:
        key_rows.append([c, MEANING[role], round(float(df[c].isna().mean() * 100), 1), int(df[c].nunique())])
for c in SKIN:
    key_rows.append([c, "Skin-type flag column provided in the dataset (0/1)",
                     round(float(df[c].isna().mean() * 100), 1), int(df[c].nunique())])
R["key_fields"] = key_rows
pd.DataFrame(key_rows, columns=["field", "meaning", "missing_%", "distinct"])

In [ ]:
# A few representative records for the report (ingredient text shortened)
show_roles = [COL[k] for k in ["name", "brand", "category", "price", "rating", "ingredients"] if COL[k]]
sample = df[show_roles].head(4).copy()
sample[COL["ingredients"]] = sample[COL["ingredients"]].astype(str).str.slice(0, 48) + "..."
R["sample"] = {"columns": show_roles,
               "rows": sample.astype(object).where(sample.notna(), None).values.tolist()}
sample

### Parsing the ingredient text, missing values and duplicates

In [ ]:
def parse_ingredients(x):
    """Ingredient text -> list of lower-case ingredient strings (handles 'a, b, c' and "['a', 'b']")."""
    if not isinstance(x, str):
        return []
    s = x.strip()
    if s.startswith("["):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, (list, tuple)):
                return [re.sub(r"\s+", " ", str(t)).strip().strip(".").lower() for t in v if str(t).strip()]
        except Exception:
            pass
    toks = [re.sub(r"\s+", " ", t).strip().strip(".").lower() for t in s.split(",")]
    return [t for t in toks if t]

tokens = df[COL["ingredients"]].map(parse_ingredients)
n_ingr = tokens.map(len)

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1),
}).sort_values("missing_pct", ascending=False)
R["missing_top"] = [[str(c), int(r.missing_count), float(r.missing_pct)] for c, r in missing.head(6).iterrows()]
R["all_null_cols"] = [str(c) for c in missing.index if missing.loc[c, "missing_count"] == n_rows]

key_cols = [c for c in [COL["brand"], COL["name"]] if c]
R["dups"] = {"full": int(df.duplicated().sum()),
             "brand_name": int(df.duplicated(subset=key_cols).sum()) if len(key_cols) == 2 else None}
print(R["dups"])
missing.head(12)

### Descriptive statistics and the data-quality issue (ingredient text)

In [ ]:
num = pd.DataFrame({"price": df[COL["price"]]})
if COL["rating"]:
    num["rating"] = df[COL["rating"]]
num["n_ingredients"] = n_ingr
desc = num.describe().round(2)
R["describe"] = {"index": [str(i) for i in desc.index],
                 "data": {c: [float(v) for v in desc[c]] for c in desc.columns}}
desc

In [ ]:
ing_text = df[COL["ingredients"]].astype(str)
all_tokens = pd.Series([t for toks in tokens for t in toks])
vc = all_tokens.value_counts()

has_water = tokens.map(lambda t: "water" in t)
has_aqua = tokens.map(lambda t: any("aqua" in x for x in t))

R["quality"] = {
    "ingr_empty": int((n_ingr == 0).sum()),
    "ingr_short": int(((n_ingr > 0) & (n_ingr < 3)).sum()),
    "ingr_placeholder": int(ing_text.str.contains(r"visit|no info|n/a|unavailable", case=False, regex=True).sum()),
    "price_zero": int((df[COL["price"]] == 0).sum()),
    "price_missing": int(df[COL["price"]].isna().sum()),
    "price_max": float(df[COL["price"]].max()),
    "tokens_total": int(len(all_tokens)), "tokens_unique": int(len(vc)),
    "tokens_once_pct": float((vc == 1).mean() * 100),
    "tokens_special_pct": float(all_tokens.str.contains(r"[()/]").mean() * 100),
    "n_water": int(has_water.sum()), "n_aqua": int(has_aqua.sum()), "n_either": int((has_water | has_aqua).sum()),
}
print(R["quality"])

## Step 3 – Preliminary EDA
A *preliminary* filter (price > 0 and at least 3 parsed ingredients) is used for the exploration only;
the final cleaning rules are Task 2 work.

In [ ]:
MIN_INGR = 3
mask = (df[COL["price"]] > 0) & (n_ingr >= MIN_INGR)
clean = df[mask].copy()
assert len(clean), "No usable records after preliminary filtering: check CSV data and ingredient parsing."
clean["_tokens"] = tokens[mask]
clean["_n_ingr"] = n_ingr[mask]
clean["_cat"] = clean[COL["category"]].astype(str) if COL["category"] else "all"
R["clean"] = {"min_ingr": MIN_INGR, "n_clean": int(len(clean)), "n_removed": int(n_rows - len(clean)),
              "pct_removed": float((n_rows - len(clean)) / n_rows * 100)}
print(R["clean"])

# Summary table: by category
agg = {"products": (COL["price"], "size"), "median_price": (COL["price"], "median"),
       "median_n_ingredients": ("_n_ingr", "median")}
if COL["rating"]:
    agg["median_rating"] = (COL["rating"], "median")
cat_tbl = clean.groupby("_cat").agg(**agg).sort_values("products", ascending=False).round(2)
R["category_table"] = [{"category": str(i), **{k: float(v) for k, v in r.items()}} for i, r in cat_tbl.iterrows()]
cat_tbl

In [ ]:
# Ingredient frequency: share of usable products that contain each ingredient
cnt = Counter(t for toks in clean["_tokens"] for t in set(toks))
top = cnt.most_common(12)
R["top_ingredients"] = [[k, int(v), round(v / len(clean) * 100, 1)] for k, v in top]
R["vocab_ge20"] = int(sum(1 for v in cnt.values() if v >= 20))
pd.DataFrame(R["top_ingredients"], columns=["ingredient", "n_products", "share_%"])

In [ ]:
# One visualization: most frequent ingredients and median price by category
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
names = [k for k, _, _ in R["top_ingredients"]][::-1]
shares = [s for _, _, s in R["top_ingredients"]][::-1]
bars = axes[0].barh([n if len(n) <= 26 else n[:25] + "…" for n in names], shares, color="#1f3a5f")
axes[0].set_title("Ingredients found in the most products", fontsize=10)
axes[0].set_xlabel("Share of usable products (%)", fontsize=8)
axes[0].tick_params(labelsize=7)
axes[0].bar_label(bars, labels=[f"{s:.0f}%" for s in shares], fontsize=7, padding=2)
axes[0].margins(x=0.15)

ct = cat_tbl.head(8)
b2 = axes[1].bar(ct.index.astype(str), ct["median_price"], color="#1f3a5f")
axes[1].set_title("Median price by product category", fontsize=10)
axes[1].set_ylabel("Median price (as given in data)", fontsize=8)
axes[1].tick_params(axis="x", labelrotation=30, labelsize=7)
axes[1].tick_params(axis="y", labelsize=7)
axes[1].bar_label(b2, labels=[f"{v:,.0f}" for v in ct["median_price"]], fontsize=7, padding=2)
axes[1].margins(y=0.15)
fig.tight_layout()
FIG_PATH = os.path.join(OUT_DIR, "eda_figure.png")
fig.savefig(FIG_PATH, dpi=200)
plt.show()

In [ ]:
# Evidence-based observations (numbers are filled in from the tables above)
q = R["quality"]
k5 = ", ".join(k for k, _, _ in R["top_ingredients"][:5])
first, last = R["top_ingredients"][0], R["top_ingredients"][-1]
obs1 = (f"The most frequent ingredient strings are {k5}. The top one ('{first[0]}') appears in {first[2]:.0f}% of the "
        f"{len(clean):,} usable products and the 12th ({last[2]:.0f}%) is far less common. This is frequency "
        "in the catalog (what formulas contain), not popularity among consumers.")

if len(cat_tbl) > 1:
    hi_c, lo_c = cat_tbl["median_price"].idxmax(), cat_tbl["median_price"].idxmin()
    hi_p, lo_p = cat_tbl.loc[hi_c, "median_price"], cat_tbl.loc[lo_c, "median_price"]
    obs2 = (f"Price differs by product category: the median is {hi_p:,.0f} for '{hi_c}' versus {lo_p:,.0f} for "
            f"'{lo_c}' ({hi_p / lo_p:.1f}x), so price comparisons should be made within a category.") if lo_p > 0 else (f"Median price ranges from {lo_p:,.0f} to {hi_p:,.0f}; the ratio is undefined for a zero median.")
else:
    obs2 = "The dataset has a single category, so price comparisons across categories are not possible."

obs3 = (f"Ingredient text is noisy: {q['tokens_once_pct']:.0f}% of the {q['tokens_unique']:,} distinct ingredient "
        f"strings occur only once, {q['tokens_special_pct']:.0f}% of all strings contain brackets or slashes, and the "
        f"same substance appears under different names ('water' in {q['n_water']:,} products, names containing "
        f"'aqua' in {q['n_aqua']:,}). Names must be normalized (Task 2) before filtering or comparing.")
R["obs"] = [obs1, obs2, obs3]
for o in R["obs"]:
    print("-", o)

## Step 4 – Prototype of the two core interactions on the real data
1. **Filter:** skin type + ingredients to include / avoid + price limit → matching products.
2. **Compare:** pick a product → similar products in the same category with a lower price (Jaccard similarity of ingredient sets).

In [ ]:
def has_ingredient(toks, word):
    return any(word.strip().casefold() in t.casefold() for t in toks)  # exploratory substring rule; refine in Task 2

def find_products(data, skin=None, include=(), avoid=(), max_price=None, top=5):
    m = pd.Series(True, index=data.index)
    if skin and skin not in SKIN:
        raise ValueError(f"Skin-type flag {skin!r} not present in the selected dataset")
    if skin:
        m &= data[skin] == 1
    if max_price is not None:
        m &= data[COL["price"]] <= max_price
    for w in include:
        m &= data["_tokens"].map(lambda t: has_ingredient(t, w))
    for w in avoid:
        m &= ~data["_tokens"].map(lambda t: has_ingredient(t, w))
    res = data[m]
    by = [COL["rating"]] if COL["rating"] else [COL["price"]]
    return res.sort_values(by, ascending=[False] if COL["rating"] else [True]), int(m.sum())

SKIN_CHOICE = next((c for c in SKIN if c.lower() == "sensitive"), SKIN[0] if SKIN else None)
QUERY = dict(skin=SKIN_CHOICE, include=["glycerin"], avoid=["fragrance"], max_price=60)
res, n_match = find_products(clean, **QUERY)
cols1 = [c for c in [COL["name"], COL["brand"], COL["category"], COL["price"], COL["rating"]] if c] + ["_n_ingr"]
R["prototype"] = {"query": QUERY, "n_matches": n_match, "columns": cols1,
                  "rows": res[cols1].head(5).astype(object).where(res[cols1].head(5).notna(), None).values.tolist()}
print(n_match, "matching products")
res[cols1].head(5)

In [ ]:
# Product comparison: same category, lower price, highest ingredient overlap
big_cat = clean["_cat"].value_counts().index[0]
pool = clean[clean["_cat"] == big_cat]
assert len(pool), "No products available to compare"
ref = pool.sort_values(COL["price"], ascending=False).iloc[0]
ref_set = set(ref["_tokens"])

rows = []
for idx, r in pool[pool[COL["price"]] < ref[COL["price"]]].iterrows():
    s = set(r["_tokens"])
    union = len(ref_set | s)
    if union:
        rows.append((r[COL["name"]] if COL["name"] else idx, r[COL["brand"]] if COL["brand"] else "-",
                     float(r[COL["price"]]), len(ref_set & s) / union, len(ref_set & s)))
rows.sort(key=lambda x: -x[3])
R["compare"] = {"category": str(big_cat), "n_candidates": len(rows),
                "ref": [str(ref[COL["name"]]) if COL["name"] else str(ref.name),
                        str(ref[COL["brand"]]) if COL["brand"] else "-", float(ref[COL["price"]]),
                        len(ref_set)],
                "rows": [list(r) for r in rows[:3]]}
print(R["compare"]["ref"])
pd.DataFrame(rows[:3], columns=["name", "brand", "price", "jaccard", "shared"])

## Optional – quick look at candidate B (for the comparison section)

In [ ]:
R["b_stats"] = None
try:
    dfb, pb = load_kaggle_csv(CONFIG["dataset_b"]["kaggle_slug"], "COSMETICS_B_CSV")
    ing_b = next((c for c in dfb.columns if "ingred" in c.lower()), None)
    R["b_stats"] = {"n_rows": int(dfb.shape[0]), "n_cols": int(dfb.shape[1]),
                    "ingr_missing_pct": float(dfb[ing_b].isna().mean() * 100) if ing_b else None}
    print(R["b_stats"], list(dfb.columns))
except Exception as e:
    print("Candidate B could not be loaded (the report will use the CONFIG text only):", repr(e)[:200])

## Save results and build the PDF report

In [ ]:
def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return str(o)

with open(os.path.join(OUT_DIR, "results.json"), "w", encoding="utf-8") as f:
    json.dump(R, f, default=_json_default, ensure_ascii=False, indent=1)
cat_tbl.to_csv(os.path.join(OUT_DIR, "summary_by_category.csv"))
pd.DataFrame(R["top_ingredients"], columns=["ingredient", "n_products", "share_pct"]).to_csv(
    os.path.join(OUT_DIR, "top_ingredients.csv"), index=False)

from build_report import build_report
pdf = build_report(R, FIG_PATH, os.path.join(OUT_DIR, "Task1_Report.pdf"))
print("Report written to", pdf)
print("IMPORTANT: Verify Kaggle licenses, team details and repository URL before submission.")
